# Full out of domain check

In [1]:
# %%
import os
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score,
)
from scipy.stats import pearsonr
from catboost import CatBoostRegressor, CatBoostClassifier, Pool
from openai import OpenAI

warnings.filterwarnings('ignore')

# ---- пути ----
BASE_PATH = "/Users/egorilin/Desktop/COMA"
BASE = Path(BASE_PATH)
EXT_OOD_DIR = BASE / 'extended_ood'
EXT_OOD_DIR.mkdir(exist_ok=True)
(EXT_OOD_DIR / 'zero_shot_preds').mkdir(exist_ok=True)

# ---- общие константы ----
RANDOM_STATE = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]
SHORT_NAMES = ['SA_MIC', 'SA_MBC', 'EC_MIC', 'EC_MBC']
OOD_COL_MAP = {
    'SA_MIC': 'MIC Sa',
    'SA_MBC': 'MBC Sa',
    'EC_MIC': 'MIC Ec',
    'EC_MBC': 'MBC Ec',
}

# Дескрипторные фичи (если есть mmff94_dipole — оставляем, иначе ниже отфильтруем)
FEATURE_COLS = [
    'Molecular weight', 'LogP',
    'mmff94_dipole', 'gasteiger_dipole', 'eem2015bm_dipole',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]

CB_KW_REG = dict(iterations=500, learning_rate=0.05, depth=6,
                 verbose=False, random_seed=RANDOM_STATE)
CB_KW_CLS = dict(iterations=500, learning_rate=0.05, depth=6,
                 verbose=False, random_seed=RANDOM_STATE,
                 auto_class_weights='Balanced')

# ---- OpenRouter (LLM эмбеддинги) ----
OPENROUTER_API_KEY = "[REDACTED_API_KEY]"
EMBEDDING_MODEL = "openai/text-embedding-3-small"
LLM_DIM = 256

_client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")

def get_embedding(text, dimensions=LLM_DIM):
    text = str(text).replace("\n", " ")[:8000]
    r = _client.embeddings.create(input=[text], model=EMBEDDING_MODEL, dimensions=dimensions)
    return np.array(r.data[0].embedding, dtype=np.float32)

def get_embeddings_batch(texts, dimensions=LLM_DIM, batch_size=50):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t).replace("\n", " ")[:8000] for t in texts[i:i+batch_size]]
        try:
            r = _client.embeddings.create(input=batch, model=EMBEDDING_MODEL, dimensions=dimensions)
            all_embs.extend([np.array(d.embedding, dtype=np.float32) for d in r.data])
        except Exception:
            for t in batch:
                try:    all_embs.append(get_embedding(t, dimensions))
                except: all_embs.append(np.zeros(dimensions, dtype=np.float32))
    return np.array(all_embs)

print(f'✓ OpenRouter, LLM_DIM={LLM_DIM}')

Device: cpu
✓ OpenRouter, LLM_DIM=256


In [2]:
# %%
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 256],
                 projection_dim=128, dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for hd in hidden_dims:
            layers.extend([nn.Linear(prev, hd), nn.BatchNorm1d(hd), nn.ReLU(), nn.Dropout(dropout)])
            prev = hd
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim), nn.BatchNorm1d(projection_dim),
            nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim), nn.BatchNorm1d(projection_dim),
        )

    def forward(self, x, return_projection=True):
        f = self.encoder(x)
        if return_projection:
            return f, self.projection_head(f)
        return f

In [3]:
# %%
OOD_MOLFORMER_PATH = f'{BASE_PATH}/Test_molformer_ood.xlsx'

if os.path.exists(OOD_MOLFORMER_PATH):
    print(f'✓ Уже существует: {OOD_MOLFORMER_PATH}')
else:
    print('Генерация molformer для OOD молекул...')
    ood_raw_tmp = pd.read_excel(f'{BASE_PATH}/QACs_SMILES_MIC_MBC.xlsx')
    smiles_list = list(ood_raw_tmp['SMILES'])

    # workaround timm/torchvision
    _timm = sys.modules.pop('timm', 'NOT_LOADED')
    _had_timm = _timm != 'NOT_LOADED'
    try:
        from transformers import AutoTokenizer, AutoModel
        tok = AutoTokenizer.from_pretrained('ibm/MoLFormer-XL-both-10pct', trust_remote_code=True)
        mf  = AutoModel.from_pretrained('ibm/MoLFormer-XL-both-10pct', trust_remote_code=True)
    finally:
        if _had_timm: sys.modules['timm'] = _timm

    mf.eval()
    inp = tok(smiles_list, padding=True, return_tensors='pt')
    with torch.no_grad():
        out = mf(**inp).pooler_output.numpy()
    pd.DataFrame(out, columns=[f'Molformer_{j}' for j in range(768)]).to_excel(OOD_MOLFORMER_PATH, index=True)
    print(f'✓ Сохранено: {OOD_MOLFORMER_PATH}')

✓ Уже существует: /Users/egorilin/Desktop/COMA/Test_molformer_ood.xlsx


In [4]:
# %%
# ---- ORIG (in-domain) ----
data_orig = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(['Unnamed: 0'], axis=1)
target_orig = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')

# desc_df — таблица дескрипторов. Если используешь файл с эмбеддингами от MolFormer,
# подставь свой путь. Здесь мы соберём desc_df на лету из data_orig + target_orig.
# Molformer_* — берём из data_orig (768 фич); MW/LogP — из target_orig если есть.
# Диполи: если нет — оставим 0.
print(f'data_orig:   {data_orig.shape}')
print(f'target_orig: {target_orig.shape}')

# попытка взять диполи из отдельного файла
DIPOLE_FILE = f'{BASE_PATH}/Train_dipole_moments.csv'
DIPOLE_COLS = ['mmff94_dipole', 'gasteiger_dipole', 'eem2015bm_dipole']

if os.path.exists(DIPOLE_FILE):
    dipoles_df = pd.read_csv(DIPOLE_FILE)
    # отбросить служебную первую колонку
    fc = dipoles_df.columns[0]
    if dipoles_df[fc].dtype.kind in 'iu' or fc.lower() in ('unnamed: 0', 'index', 'id', '№'):
        dipoles_df = dipoles_df.drop(columns=[fc])
    have_dip = [c for c in DIPOLE_COLS if c in dipoles_df.columns]
    print(f'  диполи: {have_dip} (rows={len(dipoles_df)})')
else:
    dipoles_df = None
    have_dip = []
    print('  ⚠ файла с диполями нет — будут проставлены нули')

# собираем desc_df (orig)
desc_df = pd.DataFrame(index=range(len(data_orig)))
# MW и LogP — из target_orig если есть
for c in ['Molecular weight', 'LogP']:
    if c in target_orig.columns and len(target_orig) >= len(data_orig):
        desc_df[c] = pd.to_numeric(target_orig[c].iloc[:len(data_orig)].values, errors='coerce').astype(float)
    else:
        desc_df[c] = 0.0
        print(f'  ⚠ {c} отсутствует в target_orig — поставим 0')

# диполи
for c in DIPOLE_COLS:
    if dipoles_df is not None and c in dipoles_df.columns:
        n = min(len(desc_df), len(dipoles_df))
        desc_df[c] = 0.0
        desc_df.loc[:n-1, c] = pd.to_numeric(dipoles_df[c].iloc[:n].values, errors='coerce')
    else:
        desc_df[c] = 0.0

# Molformer_* — из data_orig
for c in FEATURE_COLS:
    if c.startswith('Molformer_'):
        if c in data_orig.columns:
            desc_df[c] = pd.to_numeric(data_orig[c].values, errors='coerce').astype(float)
        else:
            desc_df[c] = 0.0

desc_df = desc_df.fillna(0.0)
print(f'desc_df:     {desc_df.shape}  ({len(FEATURE_COLS)} фич)')

# Таргеты в desc_df
for col in TARGET_COLUMNS:
    if col in target_orig.columns:
        n = min(len(desc_df), len(target_orig))
        desc_df[col] = np.nan
        desc_df.loc[:n-1, col] = pd.to_numeric(target_orig[col].iloc[:n].values, errors='coerce')

# ---- OOD ----
ood_raw = pd.read_excel(f'{BASE_PATH}/QACs_SMILES_MIC_MBC.xlsx')
ood_molformer = pd.read_excel(OOD_MOLFORMER_PATH)
for c in ['Unnamed: 0']:
    if c in ood_molformer.columns: ood_molformer = ood_molformer.drop(columns=[c])
ood_molformer = ood_molformer.select_dtypes(include=[np.number])
ood_raw['MBC Ec'] = pd.to_numeric(ood_raw['MBC Ec'], errors='coerce')

print(f'\nOOD:           {len(ood_raw)} молекул')
print(f'ood_molformer: {ood_molformer.shape}')
for sn in SHORT_NAMES:
    col = OOD_COL_MAP[sn]
    n_valid = ood_raw[col].notna().sum()
    print(f'  {sn} ({col}): {n_valid} valid')

data_orig:   (108, 768)
target_orig: (108, 10)
  диполи: [] (rows=108)
desc_df:     (108, 15)  (15 фич)

OOD:           17 молекул
ood_molformer: (17, 768)
  SA_MIC (MIC Sa): 17 valid
  SA_MBC (MBC Sa): 17 valid
  EC_MIC (MIC Ec): 17 valid
  EC_MBC (MBC Ec): 16 valid


In [6]:
# %%
with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
    scaler_clf = pickle.load(f)

encoder = AdvancedContrastiveEncoder(input_dim=768, hidden_dims=[512, 384, 256],
                                     projection_dim=128, dropout=0.3).to(device)
encoder.load_state_dict(torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth', map_location=device))
encoder.eval()

orig_scaled = scaler_clf.transform(data_orig)
ood_scaled  = scaler_clf.transform(ood_molformer)

with torch.no_grad():
    orig_emb = encoder(torch.FloatTensor(orig_scaled).to(device), return_projection=False).cpu().numpy()
    ood_emb  = encoder(torch.FloatTensor(ood_scaled ).to(device), return_projection=False).cpu().numpy()

CONTR_DIM = orig_emb.shape[1]
print(f'✓ Контрастные: orig={orig_emb.shape}, ood={ood_emb.shape}, CONTR_DIM={CONTR_DIM}')

✓ Контрастные: orig=(108, 256), ood=(17, 256), CONTR_DIM=256


In [7]:
# %%
CLASS_DESCRIPTIONS = {
    0: ("Minimum Inhibitory Concentration (MIC) against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MIC is the lowest concentration "
        "that inhibits visible bacterial growth."),
    1: ("Minimum Bactericidal Concentration (MBC) against Staphylococcus aureus ATCC 43300, "
        "a methicillin-resistant Gram-positive bacterium. MBC is the lowest concentration "
        "that kills 99.9 percent of bacteria."),
    2: ("Minimum Inhibitory Concentration (MIC) against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MIC is the lowest concentration "
        "that inhibits visible bacterial growth."),
    3: ("Minimum Bactericidal Concentration (MBC) against Escherichia coli ATCC 25922, "
        "a Gram-negative reference strain. MBC is the lowest concentration "
        "that kills 99.9 percent of bacteria."),
}

print('LLM эмбеддинги классов...')
class_emb = {i: get_embedding(d) for i, d in CLASS_DESCRIPTIONS.items()}
for i, sn in enumerate(SHORT_NAMES):
    print(f'  {sn}: dim={len(class_emb[i])}')

# SMILES для orig
smiles_col_orig = None
for c in target_orig.columns:
    if 'smiles' in c.lower():
        smiles_col_orig = c; break

if smiles_col_orig is None and 'General formula' in target_orig.columns:
    smiles_col_orig = 'General formula'

if smiles_col_orig:
    smi_list = target_orig[smiles_col_orig].astype(str).iloc[:len(data_orig)].tolist()
    print(f'\nLLM SMILES orig (col="{smiles_col_orig}", N={len(smi_list)})...')
    llm_mol_orig = get_embeddings_batch(smi_list)
else:
    print('\n⚠ SMILES в target_orig не найден — llm_mol_orig=None')
    llm_mol_orig = None

print('\nLLM SMILES OOD...')
llm_mol_ood = get_embeddings_batch(ood_raw['SMILES'].astype(str).tolist())
print(f'✓ llm_mol_orig: {None if llm_mol_orig is None else llm_mol_orig.shape}')
print(f'✓ llm_mol_ood:  {llm_mol_ood.shape}')

LLM эмбеддинги классов...
  SA_MIC: dim=256
  SA_MBC: dim=256
  EC_MIC: dim=256
  EC_MBC: dim=256

LLM SMILES orig (col="General formula", N=108)...

LLM SMILES OOD...
✓ llm_mol_orig: (108, 256)
✓ llm_mol_ood:  (17, 256)


In [8]:
# %%
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    rdkit_ok = True
except ImportError:
    rdkit_ok = False

n_ood = len(ood_raw)
desc_ood_dict = {c: np.zeros(n_ood) for c in FEATURE_COLS}

if rdkit_ok:
    print('RDKit OK — считаем MW/LogP для OOD')
    for i, smi in enumerate(ood_raw['SMILES']):
        try:
            m = Chem.MolFromSmiles(str(smi))
            if m is not None:
                if 'Molecular weight' in desc_ood_dict:
                    desc_ood_dict['Molecular weight'][i] = Descriptors.MolWt(m)
                if 'LogP' in desc_ood_dict:
                    desc_ood_dict['LogP'][i] = Crippen.MolLogP(m)
        except Exception:
            pass
else:
    print('⚠ RDKit недоступен — MW/LogP в OOD = 0')

for c in FEATURE_COLS:
    if c.startswith('Molformer_') and c in ood_molformer.columns:
        desc_ood_dict[c] = ood_molformer[c].values.astype(float)

desc_ood_arr = np.column_stack([desc_ood_dict[c] for c in FEATURE_COLS])
desc_ood_arr = np.nan_to_num(desc_ood_arr, nan=0.0)

desc_orig_arr = desc_df[FEATURE_COLS].apply(pd.to_numeric, errors='coerce').fillna(0.0).values.astype(float)

molformer_orig_arr = data_orig.values.astype(float)
molformer_ood_arr  = ood_molformer.values.astype(float)

print(f'desc_orig:      {desc_orig_arr.shape}')
print(f'desc_ood:       {desc_ood_arr.shape}')
print(f'molformer_orig: {molformer_orig_arr.shape}')
print(f'molformer_ood:  {molformer_ood_arr.shape}')

RDKit OK — считаем MW/LogP для OOD
desc_orig:      (108, 15)
desc_ood:       (17, 15)
molformer_orig: (108, 768)
molformer_ood:  (17, 768)


In [9]:
# %%
ORIG_COL_MAP = {sn: TARGET_COLUMNS[i] for i, sn in enumerate(SHORT_NAMES)}

def make_dataset(mol_features, target_df, col_map, llm_mol, class_emb_dict):
    rows = []
    for ti, sn in enumerate(SHORT_NAMES):
        col = col_map[sn]
        if col not in target_df.columns:
            continue
        vals = pd.to_numeric(target_df[col], errors='coerce')
        valid_idx = np.where(vals.notna().values)[0]
        for gi in valid_idx:
            y_raw = float(vals.values[gi])
            if y_raw <= 0:
                continue
            row = {
                'task_idx': ti,
                'short':    sn,
                'global_i': int(gi),
                'y_raw':    y_raw,
                'y_log2':   float(np.log2(y_raw)),
                'mol_feat': mol_features[gi],
                'llm_cls':  class_emb_dict[ti],
            }
            if llm_mol is not None:
                row['llm_mol'] = llm_mol[gi]
            rows.append(row)
    if not rows:
        return None
    out = {
        'X_mol':    np.stack([r['mol_feat'] for r in rows]),
        'y_log2':   np.array([r['y_log2'] for r in rows]),
        'y_raw':    np.array([r['y_raw'] for r in rows]),
        'task':     np.array([r['task_idx'] for r in rows]),
        'global_i': np.array([r['global_i'] for r in rows]),
        'llm_cls':  np.stack([r['llm_cls'] for r in rows]),
        'llm_mol':  np.stack([r['llm_mol'] for r in rows]) if 'llm_mol' in rows[0] else None,
    }
    out['D_mol'] = out['X_mol'].shape[1]
    return out

REPS = {
    'molformer': {
        'orig': make_dataset(molformer_orig_arr, target_orig, ORIG_COL_MAP, llm_mol_orig, class_emb),
        'ood':  make_dataset(molformer_ood_arr,  ood_raw,    OOD_COL_MAP,  llm_mol_ood,  class_emb),
    },
    'desc': {
        'orig': make_dataset(desc_orig_arr, target_orig, ORIG_COL_MAP, llm_mol_orig, class_emb),
        'ood':  make_dataset(desc_ood_arr,  ood_raw,    OOD_COL_MAP,  llm_mol_ood,  class_emb),
    },
    'emb': {
        'orig': make_dataset(orig_emb, target_orig, ORIG_COL_MAP, llm_mol_orig, class_emb),
        'ood':  make_dataset(ood_emb,  ood_raw,    OOD_COL_MAP,  llm_mol_ood,  class_emb),
    },
}

for rep, splits in REPS.items():
    for s, ds in splits.items():
        if ds is None:
            print(f'  {rep}/{s}: EMPTY')
        else:
            print(f'  {rep}/{s}: N={len(ds["y_log2"])}, D_mol={ds["D_mol"]}, '
                  f'llm_mol={"yes" if ds["llm_mol"] is not None else "no"}')

  molformer/orig: N=432, D_mol=768, llm_mol=yes
  molformer/ood: N=67, D_mol=768, llm_mol=yes
  desc/orig: N=432, D_mol=15, llm_mol=yes
  desc/ood: N=67, D_mol=15, llm_mol=yes
  emb/orig: N=432, D_mol=256, llm_mol=yes
  emb/ood: N=67, D_mol=256, llm_mol=yes


In [10]:
# %%
APPROACHES_BASE = ['baseline', 'task_id', 'llm_cls', 'llm_mol_cls']
APPROACHES_EMB  = APPROACHES_BASE + ['hybrid']

def build_features(ds, approach):
    """Возвращает (X, cat_features_or_None) для подхода."""
    D = ds['D_mol']
    X_mol = ds['X_mol']

    if approach == 'baseline':
        return X_mol, None
    if approach == 'task_id':
        df = pd.DataFrame(X_mol, columns=[f'f_{i}' for i in range(D)])
        df['task_id'] = ds['task'].astype(str)
        return df, ['task_id']
    if approach == 'llm_cls':
        return np.hstack([X_mol, ds['llm_cls']]), None
    if approach == 'llm_mol_cls':
        if ds['llm_mol'] is None: return None, None
        return np.hstack([ds['llm_mol'], ds['llm_cls']]), None
    if approach == 'hybrid':
        if ds['llm_mol'] is None: return None, None
        return np.hstack([X_mol, ds['llm_mol'], ds['llm_cls']]), None
    raise ValueError(f'unknown approach: {approach}')


def fit_reg(X_tr, y_tr, cat=None):
    cb = CatBoostRegressor(**CB_KW_REG)
    if cat: cb.fit(Pool(X_tr, label=y_tr, cat_features=cat))
    else:   cb.fit(X_tr, y_tr)
    return cb

def predict_reg(cb, X_te, cat=None):
    if cat: return cb.predict(Pool(X_te, cat_features=cat))
    return cb.predict(X_te)


def regression_metrics(y_true, y_pred, task_ids):
    out = {
        'N':    len(y_true),
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2':   r2_score(y_true, y_pred) if len(y_true) > 2 else np.nan,
    }
    try:    out['Pearson'] = pearsonr(y_true, y_pred)[0]
    except: out['Pearson'] = np.nan

    per_class = {}
    for tid, sn in enumerate(SHORT_NAMES):
        m = task_ids == tid
        if m.sum() < 2:
            per_class[sn] = {'N': int(m.sum()), 'MAE': np.nan, 'RMSE': np.nan, 'R2': np.nan}
            continue
        per_class[sn] = {
            'N':    int(m.sum()),
            'MAE':  mean_absolute_error(y_true[m], y_pred[m]),
            'RMSE': np.sqrt(mean_squared_error(y_true[m], y_pred[m])),
            'R2':   r2_score(y_true[m], y_pred[m]) if m.sum() > 2 else np.nan,
        }
    out['per_class'] = per_class
    return out


def build_class_encoders(orig_y_raw_per_task):
    enc = {}
    for tid in range(len(SHORT_NAMES)):
        uniq = np.sort(np.unique(np.array(orig_y_raw_per_task[tid])))
        enc[tid] = uniq.tolist()
    return enc

def y_raw_to_class(y_raw, task_ids, enc):
    y_cls = np.zeros(len(y_raw), dtype=int)
    for i, (yr, tid) in enumerate(zip(y_raw, task_ids)):
        classes = enc[tid]
        if yr in classes:
            y_cls[i] = classes.index(yr)
        else:
            arr = np.array(classes, dtype=float)
            y_cls[i] = int(np.argmin(np.abs(np.log2(arr) - np.log2(yr))))
    return y_cls

def classification_metrics(y_true_cls, y_pred_cls, task_ids):
    out = {
        'N':       len(y_true_cls),
        'Acc':     (y_true_cls == y_pred_cls).mean() if len(y_true_cls) else np.nan,
        'Acc_pm1': (np.abs(y_true_cls - y_pred_cls) <= 1).mean() if len(y_true_cls) else np.nan,
        'F1w':     f1_score(y_true_cls, y_pred_cls, average='weighted', zero_division=0)
                   if len(y_true_cls) else np.nan,
    }
    per_class = {}
    for tid, sn in enumerate(SHORT_NAMES):
        m = task_ids == tid
        if m.sum() == 0:
            per_class[sn] = {'N': 0, 'Acc': np.nan, 'Acc_pm1': np.nan}
            continue
        per_class[sn] = {
            'N':       int(m.sum()),
            'Acc':     (y_true_cls[m] == y_pred_cls[m]).mean(),
            'Acc_pm1': (np.abs(y_true_cls[m] - y_pred_cls[m]) <= 1).mean(),
        }
    out['per_class'] = per_class
    return out


def slice_ds(ds, idx):
    out = {}
    for k, v in ds.items():
        if v is None or k == 'D_mol':
            out[k] = v
        elif isinstance(v, np.ndarray):
            out[k] = v[idx]
        else:
            out[k] = v
    out['D_mol'] = ds['D_mol']
    return out

def concat_ds(a, b):
    out = {}
    for k in ('X_mol','y_log2','y_raw','task','global_i','llm_cls'):
        out[k] = np.concatenate([a[k], b[k]])
    if a['llm_mol'] is not None and b['llm_mol'] is not None:
        out['llm_mol'] = np.concatenate([a['llm_mol'], b['llm_mol']])
    else:
        out['llm_mol'] = None
    out['D_mol'] = a['D_mol']
    return out

In [ ]:
# %%
def per_target_baseline_predict_reg(ds_train, ds_test):
    y_pred = np.full(len(ds_test['y_log2']), np.nan)
    for tid in range(len(SHORT_NAMES)):
        m_tr = ds_train['task'] == tid
        m_te = ds_test['task']  == tid
        if m_tr.sum() < 3 or m_te.sum() == 0: continue
        cb = fit_reg(ds_train['X_mol'][m_tr], ds_train['y_log2'][m_tr])
        y_pred[m_te] = cb.predict(ds_test['X_mol'][m_te])
    return y_pred

def run_regression(ds_train, ds_test, approach):
    if approach == 'baseline':
        return per_target_baseline_predict_reg(ds_train, ds_test)
    X_tr, cat = build_features(ds_train, approach)
    X_te, _   = build_features(ds_test,  approach)
    if X_tr is None or X_te is None:
        return np.full(len(ds_test['y_log2']), np.nan)
    cb = fit_reg(X_tr, ds_train['y_log2'], cat=cat)
    return predict_reg(cb, X_te, cat=cat)

# ----- ZERO-SHOT -----
print('=' * 80); print('ZERO-SHOT REGRESSION'); print('=' * 80)

reg_zs_records = []
reg_zs_residuals = []
reg_zs_preds_per_target = {}

for rep in REPS:
    ds_tr = REPS[rep]['orig']; ds_te = REPS[rep]['ood']
    if ds_tr is None or ds_te is None: continue
    reg_zs_preds_per_target[rep] = {}
    approaches = APPROACHES_EMB if rep == 'emb' else APPROACHES_BASE

    for approach in approaches:
        y_pred = run_regression(ds_tr, ds_te, approach)
        valid = ~np.isnan(y_pred)
        if valid.sum() < 2: continue
        y_t = ds_te['y_log2'][valid]; y_p = y_pred[valid]; tt = ds_te['task'][valid]
        m = regression_metrics(y_t, y_p, tt)

        row = {'Rep': rep, 'Approach': approach, 'Mode': 'zero_shot',
               'N': m['N'], 'MAE': m['MAE'], 'RMSE': m['RMSE'], 'R2': m['R2'], 'Pearson': m['Pearson']}
        for sn in SHORT_NAMES:
            row[f'{sn}_MAE']  = m['per_class'][sn]['MAE']
            row[f'{sn}_RMSE'] = m['per_class'][sn]['RMSE']
            row[f'{sn}_R2']   = m['per_class'][sn]['R2']
        reg_zs_records.append(row)

        for r in np.abs(y_t - y_p):
            reg_zs_residuals.append({'Rep': rep, 'Approach': approach, 'abs_residual': r})

        rep_dict = reg_zs_preds_per_target[rep].setdefault(approach, {})
        for tid, sn in enumerate(SHORT_NAMES):
            mm = tt == tid
            if mm.sum() == 0: continue
            gi = ds_te['global_i'][valid][mm]
            codes = ood_raw['Code'].values[gi] if 'Code' in ood_raw.columns else gi
            rep_dict[sn] = pd.DataFrame({
                'Code': codes,
                'y_true_log2': y_t[mm], 'y_pred_log2': y_p[mm],
                'y_true_raw':  2.0 ** y_t[mm], 'y_pred_raw': 2.0 ** y_p[mm],
                'abs_err_log2': np.abs(y_t[mm] - y_p[mm]),
            })
        print(f'  {rep:9s} | {approach:12s} | MAE={m["MAE"]:.3f}  RMSE={m["RMSE"]:.3f}  R²={m["R2"]:.3f}')

reg_zs_df = pd.DataFrame(reg_zs_records)

# ----- FEW-SHOT -----
print('\n' + '=' * 80); print('FEW-SHOT REGRESSION (50% OOD в train)'); print('=' * 80)

ood_n = len(REPS['molformer']['ood']['y_log2'])
ood_train_idx, ood_test_idx = train_test_split(
    np.arange(ood_n), test_size=0.5, random_state=RANDOM_STATE,
    stratify=REPS['molformer']['ood']['task'],
)
print(f'OOD: train={len(ood_train_idx)} test={len(ood_test_idx)}')

reg_fs_records = []
reg_fs_residuals = []

for rep in REPS:
    ds_o = REPS[rep]['orig']; ds_d = REPS[rep]['ood']
    if ds_o is None or ds_d is None: continue
    ds_d_tr = slice_ds(ds_d, ood_train_idx)
    ds_d_te = slice_ds(ds_d, ood_test_idx)
    ds_combined = concat_ds(ds_o, ds_d_tr)

    approaches = APPROACHES_EMB if rep == 'emb' else APPROACHES_BASE
    for approach in approaches:
        y_pred = run_regression(ds_combined, ds_d_te, approach)
        valid = ~np.isnan(y_pred)
        if valid.sum() < 2: continue
        y_t = ds_d_te['y_log2'][valid]; y_p = y_pred[valid]; tt = ds_d_te['task'][valid]
        m = regression_metrics(y_t, y_p, tt)

        row = {'Rep': rep, 'Approach': approach, 'Mode': 'few_shot',
               'N': m['N'], 'MAE': m['MAE'], 'RMSE': m['RMSE'], 'R2': m['R2'], 'Pearson': m['Pearson']}
        for sn in SHORT_NAMES:
            row[f'{sn}_MAE']  = m['per_class'][sn]['MAE']
            row[f'{sn}_RMSE'] = m['per_class'][sn]['RMSE']
            row[f'{sn}_R2']   = m['per_class'][sn]['R2']
        reg_fs_records.append(row)
        for r in np.abs(y_t - y_p):
            reg_fs_residuals.append({'Rep': rep, 'Approach': approach, 'abs_residual': r})
        print(f'  {rep:9s} | {approach:12s} | MAE={m["MAE"]:.3f}  RMSE={m["RMSE"]:.3f}  R²={m["R2"]:.3f}')

reg_fs_df = pd.DataFrame(reg_fs_records)
reg_zs_resid_df = pd.DataFrame(reg_zs_residuals)
reg_fs_resid_df = pd.DataFrame(reg_fs_residuals)

print('\n--- ZS summary ---');  print(reg_zs_df[['Rep','Approach','MAE','RMSE','R2']].round(3).to_string(index=False))
print('\n--- FS summary ---');  print(reg_fs_df[['Rep','Approach','MAE','RMSE','R2']].round(3).to_string(index=False))

ZERO-SHOT REGRESSION


In [ ]:
# %%
# encoders по orig
orig_y_per_task = {tid: [] for tid in range(len(SHORT_NAMES))}
for tid in range(len(SHORT_NAMES)):
    col = ORIG_COL_MAP[SHORT_NAMES[tid]]
    vals = pd.to_numeric(target_orig[col], errors='coerce').dropna().tolist()
    orig_y_per_task[tid] = vals
class_enc = build_class_encoders(orig_y_per_task)
for tid, sn in enumerate(SHORT_NAMES):
    print(f'  {sn}: {len(class_enc[tid])} классов: {class_enc[tid]}')

def per_target_baseline_classify(ds_train, ds_test, y_cls_train):
    y_pred = np.full(len(ds_test['y_log2']), -1, dtype=int)
    for tid in range(len(SHORT_NAMES)):
        m_tr = ds_train['task'] == tid
        m_te = ds_test['task']  == tid
        if m_tr.sum() < 3 or m_te.sum() == 0: continue
        cb = CatBoostClassifier(**CB_KW_CLS)
        cb.fit(ds_train['X_mol'][m_tr], y_cls_train[m_tr])
        y_pred[m_te] = cb.predict(ds_test['X_mol'][m_te]).flatten().astype(int)
    return y_pred

def run_classification(ds_train, ds_test, approach, y_cls_train):
    if approach == 'baseline':
        return per_target_baseline_classify(ds_train, ds_test, y_cls_train)
    X_tr, cat = build_features(ds_train, approach)
    X_te, _   = build_features(ds_test,  approach)
    if X_tr is None or X_te is None:
        return np.full(len(ds_test['y_log2']), -1, dtype=int)
    cb = CatBoostClassifier(**CB_KW_CLS)
    if cat:
        cb.fit(Pool(X_tr, label=y_cls_train, cat_features=cat))
        return cb.predict(Pool(X_te, cat_features=cat)).flatten().astype(int)
    cb.fit(X_tr, y_cls_train)
    return cb.predict(X_te).flatten().astype(int)

# ----- ZERO-SHOT -----
print('=' * 80); print('ZERO-SHOT CLASSIFICATION'); print('=' * 80)
cls_zs_records = []

for rep in REPS:
    ds_tr = REPS[rep]['orig']; ds_te = REPS[rep]['ood']
    if ds_tr is None or ds_te is None: continue
    y_cls_tr = y_raw_to_class(ds_tr['y_raw'], ds_tr['task'], class_enc)
    y_cls_te = y_raw_to_class(ds_te['y_raw'], ds_te['task'], class_enc)
    approaches = APPROACHES_EMB if rep == 'emb' else APPROACHES_BASE
    for approach in approaches:
        y_pred = run_classification(ds_tr, ds_te, approach, y_cls_tr)
        valid = y_pred >= 0
        if valid.sum() < 2: continue
        m = classification_metrics(y_cls_te[valid], y_pred[valid], ds_te['task'][valid])
        row = {'Rep': rep, 'Approach': approach, 'Mode': 'zero_shot',
               'N': m['N'], 'Acc': m['Acc'], 'Acc_pm1': m['Acc_pm1'], 'F1w': m['F1w']}
        for sn in SHORT_NAMES:
            row[f'{sn}_Acc']     = m['per_class'][sn]['Acc']
            row[f'{sn}_Acc_pm1'] = m['per_class'][sn]['Acc_pm1']
        cls_zs_records.append(row)
        print(f'  {rep:9s} | {approach:12s} | Acc={m["Acc"]:.3f}  Acc±1={m["Acc_pm1"]:.3f}  F1w={m["F1w"]:.3f}')

cls_zs_df = pd.DataFrame(cls_zs_records)

# ----- FEW-SHOT -----
print('\n' + '=' * 80); print('FEW-SHOT CLASSIFICATION'); print('=' * 80)
cls_fs_records = []

for rep in REPS:
    ds_o = REPS[rep]['orig']; ds_d = REPS[rep]['ood']
    if ds_o is None or ds_d is None: continue
    ds_d_tr = slice_ds(ds_d, ood_train_idx)
    ds_d_te = slice_ds(ds_d, ood_test_idx)
    ds_combined = concat_ds(ds_o, ds_d_tr)
    y_cls_tr = y_raw_to_class(ds_combined['y_raw'], ds_combined['task'], class_enc)
    y_cls_te = y_raw_to_class(ds_d_te['y_raw'], ds_d_te['task'], class_enc)
    approaches = APPROACHES_EMB if rep == 'emb' else APPROACHES_BASE
    for approach in approaches:
        y_pred = run_classification(ds_combined, ds_d_te, approach, y_cls_tr)
        valid = y_pred >= 0
        if valid.sum() < 2: continue
        m = classification_metrics(y_cls_te[valid], y_pred[valid], ds_d_te['task'][valid])
        row = {'Rep': rep, 'Approach': approach, 'Mode': 'few_shot',
               'N': m['N'], 'Acc': m['Acc'], 'Acc_pm1': m['Acc_pm1'], 'F1w': m['F1w']}
        for sn in SHORT_NAMES:
            row[f'{sn}_Acc']     = m['per_class'][sn]['Acc']
            row[f'{sn}_Acc_pm1'] = m['per_class'][sn]['Acc_pm1']
        cls_fs_records.append(row)
        print(f'  {rep:9s} | {approach:12s} | Acc={m["Acc"]:.3f}  Acc±1={m["Acc_pm1"]:.3f}  F1w={m["F1w"]:.3f}')

cls_fs_df = pd.DataFrame(cls_fs_records)

In [ ]:
# %%
order = []
for rep in ['molformer', 'desc', 'emb']:
    approaches = APPROACHES_EMB if rep == 'emb' else APPROACHES_BASE
    for a in approaches:
        order.append((rep, a))

fig, axes = plt.subplots(2, 1, figsize=(max(14, len(order) * 0.55), 11), sharex=True)
color_map = {'molformer': '#4C72B0', 'desc': '#DD8452', 'emb': '#55A868'}

for ax, df, title in [
    (axes[0], reg_zs_resid_df, 'Zero-Shot — |residuals| (log₂)'),
    (axes[1], reg_fs_resid_df, 'Few-Shot — |residuals| (log₂)'),
]:
    data = []; labels = []; cols = []
    for rep, approach in order:
        sub = df[(df['Rep'] == rep) & (df['Approach'] == approach)]
        data.append(sub['abs_residual'].values if len(sub) >= 2 else np.array([np.nan]))
        labels.append(f'{rep}\n{approach}')
        cols.append(color_map[rep])
    parts = ax.violinplot(data, showmeans=True, showmedians=True, widths=0.85)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(cols[i]); body.set_edgecolor('black'); body.set_alpha(0.6)
    ax.set_xticks(np.arange(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('|y_true − y_pred|  (log₂)', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

axes[0].legend(handles=[
    Patch(facecolor='#4C72B0', edgecolor='black', alpha=0.6, label='molformer'),
    Patch(facecolor='#DD8452', edgecolor='black', alpha=0.6, label='desc'),
    Patch(facecolor='#55A868', edgecolor='black', alpha=0.6, label='emb'),
], loc='upper right', fontsize=10)

plt.tight_layout()
out_path = EXT_OOD_DIR / 'violin_residuals_zs_fs.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'✓ {out_path}')

In [ ]:
# %%
preds_dir = EXT_OOD_DIR / 'zero_shot_preds'
for rep, app_dict in reg_zs_preds_per_target.items():
    for approach, sn_dict in app_dict.items():
        for sn, df in sn_dict.items():
            df.to_csv(preds_dir / f'{rep}__{approach}__{sn}.csv', index=False)

reg_zs_df.to_csv(EXT_OOD_DIR / 'regression_zero_shot.csv', index=False)
reg_fs_df.to_csv(EXT_OOD_DIR / 'regression_few_shot.csv',  index=False)
cls_zs_df.to_csv(EXT_OOD_DIR / 'classification_zero_shot.csv', index=False)
cls_fs_df.to_csv(EXT_OOD_DIR / 'classification_few_shot.csv',  index=False)
reg_zs_resid_df.to_csv(EXT_OOD_DIR / 'regression_zs_residuals_long.csv', index=False)
reg_fs_resid_df.to_csv(EXT_OOD_DIR / 'regression_fs_residuals_long.csv', index=False)

excel_path = EXT_OOD_DIR / 'OOD_extended_SUMMARY.xlsx'
with pd.ExcelWriter(excel_path) as w:
    reg_zs_df.to_excel(w, sheet_name='reg_zero_shot', index=False)
    reg_fs_df.to_excel(w, sheet_name='reg_few_shot',  index=False)
    cls_zs_df.to_excel(w, sheet_name='cls_zero_shot', index=False)
    cls_fs_df.to_excel(w, sheet_name='cls_few_shot',  index=False)
    reg_zs_resid_df.to_excel(w, sheet_name='reg_zs_residuals', index=False)
    reg_fs_resid_df.to_excel(w, sheet_name='reg_fs_residuals', index=False)

    for metric, df_src, lbl in [
        ('MAE',     reg_zs_df, 'reg_zs_'),
        ('MAE',     reg_fs_df, 'reg_fs_'),
        ('RMSE',    reg_zs_df, 'reg_zs_'),
        ('RMSE',    reg_fs_df, 'reg_fs_'),
        ('R2',      reg_zs_df, 'reg_zs_'),
        ('R2',      reg_fs_df, 'reg_fs_'),
        ('Acc',     cls_zs_df, 'cls_zs_'),
        ('Acc',     cls_fs_df, 'cls_fs_'),
        ('Acc_pm1', cls_zs_df, 'cls_zs_'),
        ('Acc_pm1', cls_fs_df, 'cls_fs_'),
    ]:
        if df_src.empty: continue
        pv = df_src.pivot_table(index='Rep', columns='Approach', values=metric, aggfunc='first')
        pv.to_excel(w, sheet_name=f'{lbl}{metric}_pivot')

print(f'✓ {excel_path}')

print('\n=== Reg ZS — RMSE pivot ===')
print(reg_zs_df.pivot_table(index='Rep', columns='Approach', values='RMSE', aggfunc='first').round(3).to_string())
print('\n=== Reg FS — RMSE pivot ===')
print(reg_fs_df.pivot_table(index='Rep', columns='Approach', values='RMSE', aggfunc='first').round(3).to_string())
print('\n=== Cls ZS — Acc±1 pivot ===')
print(cls_zs_df.pivot_table(index='Rep', columns='Approach', values='Acc_pm1', aggfunc='first').round(3).to_string())
print('\n=== Cls FS — Acc±1 pivot ===')
print(cls_fs_df.pivot_table(index='Rep', columns='Approach', values='Acc_pm1', aggfunc='first').round(3).to_string())

print('\n=== Reg ZS — RMSE per class ===')
print(reg_zs_df[['Rep','Approach'] + [f'{sn}_RMSE' for sn in SHORT_NAMES]].round(3).to_string(index=False))
print('\n=== Cls ZS — Acc±1 per class ===')
print(cls_zs_df[['Rep','Approach'] + [f'{sn}_Acc_pm1' for sn in SHORT_NAMES]].round(3).to_string(index=False))

print('\n✨ ГОТОВО!')